In [2]:
from os.path import join, dirname
import os
import sys
import random
from hydra import initialize, compose
from omegaconf import DictConfig
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict
from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

from memoryPerceiver.memory_perceiver.main.configuration import cf
from memoryPerceiver.memory_perceiver.isaac.agent import create_model
import memoryPerceiver.memory_perceiver.isaac.trainer as tr
from memoryPerceiver.memory_perceiver.isaac.plot import Plotter

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


/isaac-sim/kit/python/lib/python3.7/site-packages/IPython/extensions/autoreload.py:121: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  from imp import reload


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
random.seed(0)
np.random.seed(0)
torch.use_deterministic_algorithms(False)
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


torch.Size([8, 1]) torch.Size([8]) torch.Size([8])


In [ ]:
model = create_model(device)
model_student = create_model(device)
actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))
print(actions.size(), logs.size(), vals.size())

In [8]:
"""
vectorize: add mem dict, deepcopy history -> mem, add action, vectorize death
do not care about memory being the same, backpropagate such that score are good
"""
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):
    cfg_dict = omegaconf_to_dict(cfg)

    render = True
    env = VecEnvRLGames(headless=cfg.headless, sim_device=cfg.device_id)
    
    # !!! Load isaac modules after the variables
    from omni.isaac.core.utils.torch.maths import set_seed
    import omni.usd
    # from omni.isaac.core.utils.stage import open_stage, save_stage
    
    plotter = Plotter(['loss_actor', 'loss_critic', 'rewards'])
    history = { 'reward':[], 'action':[], 'values': [], 'logprobs':[], 'generator':[]}
    previous_history = history.deepcopy()
    buffer = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[]}
    memories = {i:None for i in range(numEnvs)}
    death_index = {i:[] for i in range(numEnvs)}

    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed  # sets seed. if seed is -1 will pick a random one
    task = initialize_task(cfg_dict, env)
    all_losses_actor, all_losses_critic, all_returns = [], [], []
    optimizer = torch.optim.Adam(model_student.parameters(), lr=learning_rate)
    sim_count, sim_step = 0, 0
    first_run=True
    is_inference=False  # Will switch to True immediately
    
    while env._simulation_app.is_running():
        if env._world.is_playing():
            # if env._world.current_time_step_index == 0 or \
            # env._world.current_time_step_index % sim_duration == 0:
            if env.sim_frame_count % sim_duration == 0:

                ### Plots

                if sim_count % 6 == 1 and len(all_losses_actor)>0:
                    plotter.plot_all([(all_losses_actor, 160), (all_losses_critic, 160), all_returns])

                ### Initialization

                env._world.reset(soft=False)
                is_inference = not is_inference
                env._task.post_reset()
                print('$$$$$$$$$$$ CUMUL REWARD $$$$$$$$$$$')
                if len(history['reward'])>0:
                    print(history['reward'].sum())
                if is_inference:
                    print('PLAY')
                    if first_run and os.path.exists(path_model):
                        model.load_state_dict(torch.load(path_model))
                        model_student.load_state_dict(torch.load(path_model))
                    else:
                        if sim_count % student_update_step*2 == 0:
                            model.load_state_dict(model_student.state_dict())
                    # model.reset_memories()
                    torch.save(model_student.state_dict(), path_model)
                    model.eval()
                    death_index = {i:[] for i in range(numEnvs)}
                    generator_mem = torch.zeros(sim_duration, 816, dtype=torch.uint8)
                else:
                    print('REPLAY')
                    first_run=False
                    # model.reset_memories()
                    # model_student.reset_memories()
                    model_student.train() 
                    optimizer.zero_grad()
                    previous_history = history.deepcopy()
                    # mem_values = torch.stack(history['values']) reward action logprobs
                    previous_history['returns'] = tr.calculate_returns(previous_history['rewards'], death_index, gamma=c_gamma)
                    previous_history['advantages'] = tr.calculate_gae(previous_history['rewards'], previous_history['values'], 
                                                        death_index, gamma=c_gamma, gae_lambda=c_lambda)
                    all_returns.append(previous_history['returns'][1, :].mean().item())
                    
                torch.manual_seed(sim_count//2)
                torch.cuda.manual_seed(sim_count//2)
                memories = {i:None for i in range(numEnvs)}
                birth_time = torch.zeros(numEnvs)
                for key in buffer.keys():
                    buffer[key]=[]
                for key in history.keys():
                    history[key]=[]
                backward_count, kl_div = 0, 0
                sim_count+=1
                sim_step = 0

            ### Get agent observations 

            all_obs = env._task.get_observations()['cartpole_view']
            obs = all_obs['obs_buf'].clone()    # (numEnvs, 4)
            video = all_obs['video_buf'].clone()  # (numEnvs, 3, 256, 456)
            env._task.calculate_metrics()       
            reward=env._task.rew_buf.clone()    # (numEnvs, 1)
            tr.update_buffers(buffer, {'obs': obs, 'video': video, 'reward': reward})
            # update history obs, reward


            ### Reset goal and memory at death

            for agent_i in range(numEnvs):
                if env._task.reset_buf[agent_i].item() == 1:
                    memories[agent_i] = None
                    birth_time[agent_i] = sim_step

                    # Update death dictionnary during inference

                    if is_inference:
                        death_index[agent_i].append(sim_step)


            ### Process input

            inputs={}
            inputs['video'] = torch.zeros(numEnvs, 3, cf.frames, 256, 456, device=device)
            # inputs['video'] = rearrange(buffer_padding(buffer['video'], [3, 256, 456]), 'b t c h w -> b c t h w')
            inputs['audio'] = torch.zeros(numEnvs, cf.frames*1920, device=device)

            ### Agent
            
            if len(buffer['action'])==0 :
                inputs['observations'] = tr.buffer_padding(torch.stack(buffer['obs']), [cf.body_parts['observations']])
                # mask = create_mask(birth_time, sim_step)
                # inputs['observations'][mask] *= 0
                dt=3
                state = inputs['observations'][:,-1]


                ## Inference and memory update

                if is_inference:
                    actions, logprobs, values, old_dist = model.act(state)                    
                    history['generator'][sim_step] = torch.cuda.get_rng_state().clone()
                else:
                    actions = rearrange( tr.buffer_padding(previous_history['actions'][sim_step: sim_step+1], [1], frames=1), 'b 1 d -> b d')
                    logprobs = rearrange( tr.buffer_padding(previous_history['logprobs'][sim_step: sim_step+1], [], frames=1),  'b 1 -> b')
                    values = rearrange( tr.buffer_padding(previous_history['values'][sim_step: sim_step+1], [], frames=1),  'b 1 -> b')
                    torch.cuda.set_rng_state(previous_history['generator'][sim_step])
                                    

                ## Backward pass

                if not is_inference and sim_step>0:

                    # returns = rearrange( buffer_padding(returns_mem[sim_step:sim_step+dt], [], frames=1) , 'b 1 -> b')
                    # advantages = rearrange( buffer_padding(advantages_mem[sim_step:sim_step+dt], [], frames=1), 'b 1 -> b')
                    returns = tr.buffer_padding(previous_history['return'][sim_step:sim_step+dt], [], frames=dt).mean(dim=-1)
                    advantages = tr.buffer_padding(previous_history['advantage'][sim_step:sim_step+dt], [], frames=dt).mean(dim=-1)
                    new_logprobs, new_values, entropies, new_dist = model_student.evaluate(state, actions)

                    # Loss
                    ratios = torch.exp(new_logprobs - logprobs)
                    surr1 = ratios * advantages
                    surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantages
                    loss_actor = -(torch.min(surr1, surr2)).mean()
                    loss_entropy = -c_entropy * entropies.mean()
                    loss_critic = c_critic * F.mse_loss(new_values, returns)
                    loss =  (loss_actor + loss_critic + loss_entropy) /accumulation_steps
                    loss.backward()
                    backward_count += 1
                    kl_div += torch.distributions.kl_divergence(new_dist, old_dist).mean()/accumulation_steps


                    # Step optimizer

                    if backward_count % accumulation_steps == 0 and backward_count != 0:
                        torch.nn.utils.clip_grad_norm_(model_student.parameters(), c_grad_norm)
                        optimizer.step()
                        optimizer.zero_grad()

                        # Adaptive schedule
                        for param_group in optimizer.param_groups:
                            if adaptive_schedule:
                                param_group['lr'] = tr.update_lr(param_group['lr'], kl_div)
                            print('lr', param_group['lr'], 'kl', kl_div.item())

                        # Reinit
                        backward_count = 0
                        kl_div = 0


                    all_losses_actor.append(loss_actor.item())
                    all_losses_critic.append(loss_critic.item())

                # Update buffers

                actions_inference = list(torch.unbind( repeat(actions, 'b d -> t b d', t=dt) ))
                buffer['action'] += actions_inference
                history['action'] += actions_inference
                history['values'] += list(torch.unbind( repeat(values, 'b -> t b', t=dt) ))
                history['logprobs'] += list(torch.unbind( repeat(logprobs, 'b -> t b', t=dt) ))


            ## Current action (numEnvs, 1)

            if(len(buffer['action'])>0):
                action = buffer['action'].pop(0)
                action = torch.clamp(action, min=-1, max=1)
            else:
                action = torch.dzeros(numEnvs, 1)

            # if( inference_done ):
            #     print('$$$$$$  INFERENCE  $$$$$$')
            #     print('obs ', inputs['observations'][agent_i,-1])
            #     if replay_training:
            #         print('advantages: ', advantages)

            # if env.sim_frame_count % sim_duration == 1:
            #     print("$$$$$$$$$$$$ 00000 $$$$$$$$$$$")
            #     if replay_training:
            #         print('returns: ', returns_tmp)

            # Step environment

            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()

            sim_step+=1

        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    sim_duration = 97
    numEnvs = 8
    learning_rate = 0.0001
    accumulation_steps = 4
    c_entropy = 0
    c_critic=4
    student_update_step = 1
    eps=1e-9
    c_gamma=0.99
    c_lambda=0.95
    c_grad_norm=1.0
    eps_clip=0.2
    adaptive_schedule=False
    path_model = 'weights/cartpole.pt'
    critic_path = 'weights/critic.pt'
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True","+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


Starting kit application with the following args:  ['/isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py', '/isaac-sim/apps/omni.isaac.sim.python.gym.headless.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=False', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--portable', '--no-window', '--allow-root']
Passing the following args to the base kit application:  ['-f', '/root/.local/share/jupyter/runtime/kernel-1ccff390-010d-4eef-809e-89d84b75c983.json']
[Warning] [omni.isaac.kit.simulation_app] Modules: ['omni.isaac.kit.app_framework'] were loaded before SimulationApp was sta

2023-10-14 13:07:15 s] [Error] [carb.glinterop.plugin] GLInteropContext::init: carb::windowing is not available
2023-10-14 13:07:15 s] [Error] [gpu.foundation.plugin] No device could be created. Some known system issues:
- The driver is not installed properly and requires a clean re-install.
- Your GPUs do not support RayTracing: DXR or Vulkan ray_tracing, or hardware is excluded due to performance.
- The driver cannot enumerate any GPU: driver, display, TCC mode or a docker issue. For Vulkan, test it with Vulkaninfo tool from Vulkan SDK, instead of nvidia-smi.
- For Ubuntu, it requires server-xorg-core 1.20.7+ and a display to work without --no-window.
- For Linux dockers, the setup is not complete. Install the latest driver, xServer and NVIDIA container runtime.

2023-10-14 13:07:15 s] [Error] [carb.glinterop.plugin] GLInteropContext::init: carb::windowing is not available


2023-10-14 13:07:15 s] [Warning] [carb.graphics-vulkan.plugin] No command queue family supports flags: 0x100, queue type: 3. No queues of this type will be created
2023-10-14 13:07:15 s] [Warning] [carb.graphics-vulkan.plugin] No command queue family supports flags: 0x100, queue type: 3. No queues of this type will be created
2023-10-14 13:07:15 s] [Warning] [carb.graphics-vulkan.plugin] No command queue family supports flags: 0x100, queue type: 3. No queues of this type will be created
2023-10-14 13:07:15 [1,035ms] [Warning] [carb.graphics-vulkan.plugin] No command queue family supports flags: 0x100, queue type: 3. No queues of this type will be created

|---------------------------------------------------------------------------------------------|
| Driver Version: 525.60.11     | Graphics API: Vulkan
|=============================================================================================|
| GPU | Name                             | Active | LDA | GPU Memory | Vendor-ID | LUID  

/isaac-sim/kit/exts/omni.graph/omni/graph/core/_impl/autonode/type_definitions.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  Float = NewType("Float", np.float)


ndowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2023-10-14 13:07:16 [1,714ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-10-14 13:07:16 [1,714ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
[1.768s] [ext: omni.kit.renderer.capture-0.0.0] startup
[1.772s] [ext: omni.kit.renderer.imgui-0.0.0] startup
2023-10-14 13:07:16 [1,728ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-10-14 13:07:16 [1,728ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2023-10-14 13:07:16 [1,729ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-10-14 13:07:16 [1,729ms] [Warning] [carb] Failed to startup p

/isaac-sim/kit/kernel/py/omni/kit/app/_impl/__init__.py:104: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  if not isinstance(setting_py, str) and isinstance(setting_py, collections.Sequence):


Warp 0.6.3 initialized:
   CUDA Toolkit: 11.5, Driver: 12.0
   Devices:
     "cpu"    | x86_64
     "cuda:0" | NVIDIA GeForce GTX 1070 (sm_61)
     "cuda:1" | NVIDIA GeForce GTX 1070 (sm_61)
     "cuda:2" | NVIDIA GeForce GTX 1070 (sm_61)
     "cuda:3" | NVIDIA GeForce GTX 1070 (sm_61)
   Kernel cache: /root/.cache/warp/0.6.3
Setting seed: 42
Sim params does not have attribute:  physx
Sim params does not have attribute:  Cartpole
Pipeline:  GPU
Pipeline Device:  cuda:0
Sim Device:  GPU
36s] [ext: omni.kit.material.library-1.3.21] startup
[3.353s] [ext: omni.kit.browser.core-2.2.2] startup
[3.364s] [ext: omni.kit.window.file_exporter-1.0.10] startup
[3.367s] [ext: omni.physics.tensors-0.1.0] startup
[3.386s] [ext: omni.kit.browser.folder.core-1.7.3] startup
[3.395s] [ext: omni.kit.window.file-1.3.32] startup
[3.401s] [ext: omni.physx.tensors-0.1.0] startup
[3.418s] [ext: omni.isaac.version-1.0.0] startup
[3.419s] [ext: omni.kit.browser.sample-1.2.5] startup
[3.423s] [ext: omni.isaac.clo

/isaac-sim/kit/python/lib/python3.7/site-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


startup
[5.689s] [ext: omni.kit.viewport.bundle-104.0.2] startup
[5.708s] [ext: omni.volume-0.1.0] startup
[5.727s] [ext: omni.hydra.rtx-0.1.0] startup
[5.746s] [ext: omni.activity.core-1.0.1] startup
[5.754s] [ext: omni.kit.window.property-1.8.2] startup
[5.773s] [ext: omni.kit.widget.prompt-1.0.5] startup
[5.779s] [ext: omni.kit.widget.graph-1.5.6-104_2] startup
[5.812s] [ext: omni.kit.widget.text_editor-1.0.2] startup
[5.815s] [ext: omni.debugdraw-0.1.1] startup
[5.831s] [ext: omni.kit.window.extensions-1.1.1] startup
[5.844s] [ext: omni.inspect-1.0.1] startup
[5.850s] [ext: omni.kit.widget.stage-2.7.24] startup
[5.861s] [ext: omni.kit.property.usd-3.18.17] startup
[5.883s] [ext: omni.graph.ui-1.24.2] startup
[5.978s] [ext: omni.graph.scriptnode-0.10.0] startup
[5.985s] [ext: omni.command.usd-1.0.2] startup
[5.990s] [ext: omni.graph.action-1.31.1] startup
[6.013s] [ext: omni.syntheticdata-0.2.4] startup
2023-10-14 13:07:20 [5,997ms] [Warning] [omni.kit.viewport.menubar.core.menu_ite